In [ ]:
%load_ext autoreload
%autoreload 2

test readers

In [ ]:
from microimage import Reader
from pathlib import Path
import xarray as xr

# define the data file
filename = Path("/media/cephfs/sankar/Images for Jerome/180925_GFP 488_PCNT 555_new_SunBicD2_No Dox control_1.lsm")

# parse the file
with Reader(filename) as f:
   for k, s in enumerate(f.iter_scenes()):
      print(k, s.attrs)
      


In [ ]:
from microimage import Reader
from pathlib import Path
import xarray as xr

# define the data file
filename = Path("/media/cephfs/sankar/SunTag images/60X Sun images/121125_GFP 488_PCNT 647_SunBicD2 FL CC1 new_multiple.lsm")

# parse the file
with Reader(filename) as f:
   for s in f.iter_scenes():
      print(s)
      assert s.dims == ("Z","C","Y","X")
      break

In [ ]:
s.attrs['unique_id']

In [ ]:

filename = "/media/cephfs/sankar/Images for Jerome/SunTag images/111125_GFP 488_PCNT 647_SunBicD2 FL CCmut new_1.czi"
# parse the file
with Reader(filename) as f:
   for s in f.iter_scenes():
      print(s)
      break



In [14]:
p(filename)

[{'scene': '0',
  '_scene_xarray': <xarray.DataArray 'from-zarr-20dae8217936eea4dd1257fa32e4cb2e' (Z: 8, C: 3,
                                                                  Y: 1584, X: 1584)> Size: 120MB
  dask.array<from-zarr, shape=(8, 3, 1584, 1584), dtype=uint16, chunksize=(1, 1, 1584, 1584), chunktype=numpy.ndarray>
  Coordinates:
    * Z        (Z) float64 64B 0.0 6.337e-07 1.267e-06 ... 3.802e-06 4.436e-06
    * C        (C) <U7 84B 'ChS1-T1' 'Ch1-T2' 'Ch2-T2'
    * Y        (Y) float64 13kB 0.0 8.52e-08 1.704e-07 ... 0.0001348 0.0001349
    * X        (X) float64 13kB 0.0 8.52e-08 1.704e-07 ... 0.0001348 0.0001349
  Attributes: (12/68)
      MagicNumber:                  67127628
      StructureSize:                464
      DimensionX:                   1584
      DimensionY:                   1584
      DimensionZ:                   8
      DimensionChannels:            3
      ...                           ...
      Positions:                    [[0. 0. 0.]]
      TilePo

In [ ]:
from microimage import ExperimentManager
from microimage import Reader

class MetadataParser:
    def __init__(self, extra_metadata=None):
        self.extra_metadata = extra_metadata
    def __call__(self,path):
        items = []
        try:
            with Reader(path) as img: # open the file
                g = str(path.stem).split("_")[5]
                for k, scene in enumerate(img.iter_scenes()):
                    items.append({
                        "scene": str(k),
                        "_scene_xarray": scene,
                        "unique_id": scene.attrs['unique_id'],
                        "metadata": {"group":g},
                    })
        except Exception as e:
            print(f"  Could not read {path.name}: {e}")
        return items



# p = parse_metadata({"dapi": 1, "nuclei": 2, "pcnt": 0,"condition":"wt"})

# define the input folder
data_dir = Path("/media/cephfs/sankar/Images for Jerome/")
# define the project/results folder
results_dir = Path("./results")
# define the database 
db_path = Path("project.db")
db_path.unlink(True)
# 
m = ExperimentManager(data_dir, results_dir, db_path)
p = MetadataParser()
# scan the directory with the parser
m.scan_data_directory(["lsm"], p)

Project root (Base for result paths) set to: /home/jeromeb/work/code/microimage
Initializing database at /home/jeromeb/work/code/microimage/project.db...
Database structure verified/created.
Scanning /media/cephfs/sankar/Images for Jerome for new files...
  Skipping hidden resource file: ._180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_multiple_2.lsm
  Using parser to analyze file: 180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_multiple_2.lsm
  Registered new item: e2a521cd90f109670abebddec6eec97924e939dd
  Registered new item: 83e815d24cda153d04650650ca896ff1d0acbbbb
  Registered new item: 8e01196fb87aaf8f6ebb2b5dd3400654ac2720ad
  Registered new item: 173bd95b0df9f6722a2e122cf440ecd55143fdb6
  Registered new item: 4d196695944eb4cb520f5884f4d3e9b9cad81aa4
  Registered new item: 51ef2705b7de35f6ac6e24da374c48b402b62fe7
  Registered new item: 2145fecb8a36367a456b16ff40c396f344eeb3d2
  Registered new item: 84ab32d0da528021e5318f7013e9223ee9649efd
  Registered new item: f2deef572fadcc4c76eab9

In [ ]:
for x in m:
    g = str(x.relative_image_path).split("_")[5]
    x.add_metadata({"group":g})
    if x.size_c == 4:
        x.add_metadata({"group":g,"pcnt":1,"g3bp1":1,"dapi":2,"rna":0})
    else:
        x.add_metadata({"group":g,"pcnt":1,"dapi":1,"rna":0})
    m.update_item(x)

In [39]:
m.to_dataframe()

,id,relative_image_path,scene,metadata,processed,error,size_t,size_c,size_z,size_y,size_x,scale_t,scale_z,scale_y,scale_x,channels
0,e2a521cd90f109670abebddec6eec97924e939dd,180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_m...,0,"{""group"": ""siG2BP1""}",0,None,None,3,6,1584,1584,None,6.336906e-07,8.519643e-08,8.519643e-08,"[""ChS1-T1"", ""Ch1-T2"", ""Ch2-T2""]"
1,83e815d24cda153d04650650ca896ff1d0acbbbb,180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_m...,1,"{""group"": ""siG2BP1""}",0,None,None,3,6,1584,1584,None,6.336906e-07,8.519643e-08,8.519643e-08,"[""ChS1-T1"", ""Ch1-T2"", ""Ch2-T2""]"
2,8e01196fb87aaf8f6ebb2b5dd3400654ac2720ad,180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_m...,2,"{""group"": ""siG2BP1""}",0,None,None,3,6,1584,1584,None,6.336906e-07,8.519643e-08,8.519643e-08,"[""ChS1-T1"", ""Ch1-T2"", ""Ch2-T2""]"
3,173bd95b0df9f6722a2e122cf440ecd55143fdb6,180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_m...,3,"{""group"": ""siG2BP1""}",0,None,None,3,6,1584,1584,None,6.336906e-07,8.519643e-08,8.519643e-08,"[""ChS1-T1"", ""Ch1-T2"", ""Ch2-T2""]"
4,4d196695944eb4cb520f5884f4d3e9b9cad81aa4,180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_m...,4,"{""group"": ""siG2BP1""}",0,None,None,3,6,1584,1584,None,6.336906e-07,8.519643e-08,8.519643e-08,"[""ChS1-T1"", ""Ch1-T2"", ""Ch2-T2""]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,ccdab4a3d64a32d2090023d7b1f898ac6f8797c9,061025_GFP 488_G3BP1 555_PCNT 647_SunBicD2_siG...,10,"{""group"": ""siG3BP1""}",0,None,None,4,20,2796,2796,None,2.579253e-07,4.826579e-08,4.826579e-08,"[""Ch1-T1"", ""ChS2-T1"", ""ChS1-T2"", ""Ch2-T2""]"
73,e8e93a48b39cc247e1a4ba8492382be1ebedf5ae,061025_GFP 488_G3BP1 555_PCNT 647_SunBicD2_siG...,11,"{""group"": ""siG3BP1""}",0,None,None,4,20,2796,2796,None,2.579253e-07,4.826579e-08,4.826579e-08,"[""Ch1-T1"", ""ChS2-T1"", ""ChS1-T2"", ""Ch2-T2""]"
74,6a521ed81142f077e6efa40e067e10e102e1d41e,061025_GFP 488_G3BP1 555_PCNT 647_SunBicD2_siG...,12,"{""group"": ""siG3BP1""}",0,None,None,4,20,2796,2796,None,2.579253e-07,4.826579e-08,4.826579e-08,"[""Ch1-T1"", ""ChS2-T1"", ""ChS1-T2"", ""Ch2-T2""]"
75,345ccfa218dbe413e5eec716c6a126cdfec50b5e,061025_GFP 488_G3BP1 555_PCNT 647_SunBicD2_siG...,13,"{""group"": ""siG3BP1""}",0,None,None,4,20,2796,2796,None,2.579253e-07,4.826579e-08,4.826579e-08,"[""Ch1-T1"", ""ChS2-T1"", ""ChS1-T2"", ""Ch2-T2""]"


In [ ]:
from ome_types.model import Channel
from ome_types.model import Image
from ome_types.model import OME
from ome_types.model import Pixels
from ome_types.model import TiffData
import uuid

# ome = OME(uuid=f"urn:uuid:{uuid.uuid4()}")
ome = OME()
pixels = Pixels(
    dimension_order='XYZCT',
    physical_size_x="0.2516",
    physical_size_y="0.2516",
    physical_size_z="0.75",
    size_x=256,
    size_y=256,
    size_z=31,
    size_c=3,
    size_t=40,
    type='uint16')
pixels.channels.extend([
  Channel(color="16711935", name="NCAPD2", samples_per_pixel=1),
  Channel(color="65535", name="DNA", samples_per_pixel=1),
  Channel(color="-1", name="NEG_Dextran", samples_per_pixel=1)])
ome.images.append(Image(name="cell0005_R0001", pixels=pixels))

ome

In [ ]:
import json

ome = OME()
pixels = Pixels(
    dimension_order='XYZCT',
    physical_size_x="0.2516",
    physical_size_y="0.2516",
    physical_size_z="0.75",
    size_x=256,
    size_y=256,
    size_z=31,
    size_c=3,
    size_t=40,
    type='uint16',
    channels=[
    Channel(color="16711935", name="NCAPD2", samples_per_pixel=1),
    Channel(color="65535", name="DNA", samples_per_pixel=1),
    Channel(color="-1", name="NEG_Dextran", samples_per_pixel=1)])
ome.images.append(Image(name="cell0005_R0001", pixels=pixels))
ome.to_xml()



In [ ]:
import xarray as xr
data = xr.DataArray([[10,10,10],[10,10,10]], dims=("y","x"))
data.attrs["long_name"] 

In [ ]:
from pylibCZIrw import czi as pyczi
filepath = "/media/cephfs/sankar/Images for Jerome/SunTag images/111125_GFP 488_PCNT 647_SunBicD2 FL CCmut new_1.czi"
with pyczi.open_czi(filepath) as czi:
    print(czi.metadata)

In [ ]:
# In a Jupyter Notebook cell

import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Assuming you have your manager and readers available
from microimage import ExperimentManager, Reader 

# # --- Setup ---
# # Create a dummy setup for demonstration
# # In your real use case, you would use your actual paths.

# # define the input folder
# data_dir = "/media/cephfs/sankar/Images for Jerome"
# # define the project/results folder
# results_dir = "./results"
# # define the database 
# db_path = "project.db"

# # --- Initialize the Manager and Scan ---
# manager = ExperimentManager(data_dir, results_dir, db_path)
# manager.scan_data_directory(extensions=[".lsm"])



In [ ]:
manager = m
# --- Create Widgets ---

# Get a list of unprocessed items for the dropdown
unprocessed_items = manager.get_unprocessed_items()
item_options = {f"{item.id}": item for item in unprocessed_items}

# Dropdown to select an item
item_selector = widgets.Dropdown(
    options={item.id: item for item in unprocessed_items},
    description='Unprocessed Items:',
    style={'description_width': 'initial'},
    layout={'width': '50%'}
)

# An output widget to display information about the selected item
info_display = widgets.Output(layout={'border': '1px solid black', 'padding': '10px'})

# A button to perform an action
process_button = widgets.Button(description="Mark as Processed")
status_label = widgets.Label() # To show feedback

# --- Define Widget Logic ---

def on_selection_change(change):
    """This function runs when a new item is selected in the dropdown."""
    if change['new']:
        selected_item = change['new']
        
        # The info_display widget can be used as a context manager to capture output
        with info_display:
            clear_output(wait=True) # Clear previous output
            print(f"ID: {selected_item.id}")
            print(f"Source File: {selected_item.relative_image_path}")
            print(f"Processed: {selected_item.processed}")
            print("\n--- Metadata ---")
            # In a real scenario, you'd pretty-print the metadata
            print(selected_item.metadata)

            # You could also display the image here using a library like matplotlib
            # from microimage import Reader
            # with Reader(manager.get_absolute_container_path(selected_item.id)) as img:
            #     scene = next(img.scenes())
            #     # ... plot scene ...
            
def on_button_click(b):
    """This function runs when the button is clicked."""
    selected_id = item_selector.value.id
    if not selected_id:
        status_label.value = "No item selected."
        return
        
    # Get the latest version of the item, update it, and save it
    item_to_update = manager[selected_id]
    item_to_update.processed = True
    item_to_update.error = None
    manager.update_item(item_to_update)
    
    status_label.value = f"'{selected_id}' marked as processed. Refresh dropdown to see changes."


# --- Link Logic to Widgets ---
item_selector.observe(on_selection_change, names='value')
process_button.on_click(on_button_click)


# --- Display the UI ---
# Display the widgets in a vertical box layout
ui = widgets.VBox([item_selector, info_display, process_button, status_label])
display(ui)

# Trigger the display for the initial selection
on_selection_change({'new': item_selector.value})



In [ ]:
# /home/jeromeb/work/code/microimage/src/microimage/ui.py
import tkinter as tk
from tkinter import ttk  # For the table (TreeView)
from pathlib import Path
from microimage import ExperimentManager  # Assuming relative import works

class ExperimentManagerUI:
    def __init__(self, data_dir, results_dir, db_path):
        self.manager = ExperimentManager(data_dir, results_dir, db_path)
        self.root = tk.Tk()
        self.root.title("Experiment Manager")
        self.tree = None  # Initialize tree to None
        self.create_widgets()
        self.populate_table()

    def create_widgets(self):
        # --- Table (TreeView) ---
        self.tree = ttk.Treeview(
            self.root,
            columns=(
                "id",
                "relative_image_path",
                "scene",
                "processed",
                "error",
                "size_t",
                "size_c",
                "size_z",
                "size_y",
                "size_x",
                "scale_t",
                "scale_z",
                "scale_y",
                "scale_x",
                "channels",
            ),
            show="headings",
        )

        # Define column headings
        self.tree.heading("id", text="ID")
        self.tree.heading("relative_image_path", text="Path")
        self.tree.heading("scene", text="Scene")
        self.tree.heading("processed", text="Processed")
        self.tree.heading("error", text="Error")
        self.tree.heading("size_t", text="Size T")
        self.tree.heading("size_c", text="Size C")
        self.tree.heading("size_z", text="Size Z")
        self.tree.heading("size_y", text="Size Y")
        self.tree.heading("size_x", text="Size X")
        self.tree.heading("scale_t", text="Scale T")
        self.tree.heading("scale_z", text="Scale Z")
        self.tree.heading("scale_y", text="Scale Y")
        self.tree.heading("scale_x", text="Scale X")
        self.tree.heading("channels", text="Channels")

        # --- Layout ---
        self.tree.pack(fill=tk.BOTH, expand=True)

        # --- Buttons ---
        scan_button = tk.Button(self.root, text="Scan Directory", command=self.scan_directory)
        scan_button.pack()

        reprocess_all_button = tk.Button(self.root, text="Reprocess All", command=self.reprocess_all)
        reprocess_all_button.pack()

    def populate_table(self):
        # Clear existing rows
        for item in self.tree.get_children():
            self.tree.delete(item)

        # Populate with data from the ExperimentManager
        for item in self.manager:
            self.tree.insert(
                "",
                tk.END,
                values=(
                    item.id,
                    item.relative_image_path,
                    item.scene,
                    item.processed,
                    item.error,
                    item.size_t,
                    item.size_c,
                    item.size_z,
                    item.size_y,
                    item.size_x,
                    item.scale_t,
                    item.scale_z,
                    item.scale_y,
                    item.scale_x,
                    item.channels,
                ),
            )

    def scan_directory(self):
        self.manager.scan_data_directory()
        self.populate_table()

    def reprocess_all(self):
        self.manager.force_reprocess_all()
        self.populate_table()

    def run(self):
        self.root.mainloop()




ui = ExperimentManagerUI(data_dir, results_dir, db_path)
# Check if data_dir and results_dir exist
if not Path(data_dir).exists():
    print(f"Error: Data directory '{data_dir}' does not exist.")

if not Path(results_dir).exists():
    print(f"Error: Results directory '{results_dir}' does not exist.")


ui.run()


